# ORASR Advanced Workflows

This notebook compares ORASR routing outcomes across a risk sweep, highlights pathway transitions, and visualizes latency and safety outcomes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from orasr import ORASRRouter

In [ ]:
router = ORASRRouter(enable_fast_path=True, enable_audit=True)
router.pathways

## Run a structured risk sweep

In [ ]:
def clinical_action(payload):
    return {'status': 'ok', 'patient_id': payload['patient_id'], 'recommended_action': payload['action']}

risk_grid = [0.05, 0.15, 0.29, 0.30, 0.45, 0.69, 0.70, 0.85, 0.95]
rows = []
for risk in risk_grid:
    result = router.route(
        action=clinical_action,
        input_data={'patient_id': f'PT-{int(risk * 100):03d}', 'action': 'review'},
        risk_score=risk,
        require_human_approval=risk >= 0.70,
        human_approved=risk >= 0.70,
    )
    rows.append({
        'risk_score': risk,
        'pathway': result.path.name,
        'safe': result.safe,
        'latency_ms': result.latency * 1000,
        'gates_passed': len(result.gates_passed),
        'violations': '; '.join(result.violations),
    })

rows

## Visualize pathway transitions and latency

In [ ]:
pathway_order = {'FAST': 1, 'NORMAL': 2, 'SAFE': 3}
x = [row['risk_score'] for row in rows]
y = [pathway_order[row['pathway']] for row in rows]
colors = ['#2a9d8f' if row['safe'] else '#d62828' for row in rows]
latencies = [row['latency_ms'] for row in rows]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(x, y, s=120, c=colors, edgecolors='black')
axes[0].plot(x, y, color='#457b9d', alpha=0.5)
axes[0].set_yticks([1, 2, 3], ['FAST', 'NORMAL', 'SAFE'])
axes[0].set_xlabel('Risk score')
axes[0].set_ylabel('Selected pathway')
axes[0].set_title('Pathway escalation by risk score')
axes[0].grid(True, alpha=0.3)

axes[1].bar([str(r) for r in x], latencies, color='#e9c46a', edgecolor='#bc8a1f')
axes[1].set_xlabel('Risk score')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Observed routing latency')
axes[1].grid(True, axis='y', alpha=0.3)

fig.tight_layout()

## Approval-sensitive scenario comparison

In [ ]:
approved = router.route(
    action=clinical_action,
    input_data={'patient_id': 'PT-APPROVED', 'action': 'urgent review'},
    risk_score=0.88,
    require_human_approval=True,
    human_approved=True,
)
rejected = router.route(
    action=clinical_action,
    input_data={'patient_id': 'PT-REJECTED', 'action': 'urgent review'},
    risk_score=0.88,
    require_human_approval=True,
    human_approved=False,
)

{
    'approved_case': {'safe': approved.safe, 'violations': approved.violations},
    'rejected_case': {'safe': rejected.safe, 'violations': rejected.violations},
}

## Aggregate router statistics

In [ ]:
router.get_statistics()